# Jupyter Notebook
**Projekt:** The Hidden Cost of Weather  
**Autor:** Lukáš Weissgráb

Tento Jupyter Notebook obsahuje podrobnou cestu datového projektu, který se týká vlivu počasí na zpoždění letů v USA.

## 01 - Data Extraction (METAR zprávy & Delays)






Importujeme důležité knihovny pro naší analýzu.

In [2]:
import os
from pathlib import Path
import time
import urllib.parse
import zipfile
import pandas as pd
import numpy as np
import requests

# Příprava složek v souladu s architekturou projektu
RAW_DIR = Path("data/raw")
RAW_DIR.mkdir(parents=True, exist_ok=True)

print(f"Úložiště připraveno: {RAW_DIR.resolve()}")

Úložiště připraveno: /content/data/raw


Nyní nás čeká import jednotlivých dat. Začneme importem METAR zpráv, což jsou zprávy týkající se počasí na daném letišti (zahrnují čas vydání, vítr, teplotu, rosný bod, dohlednost a význačné jevy počasí - déšť, bouřka apod.).

In [3]:
# -------------------------------------------------------------
# STAŽENÍ A SPOJENÍ METAR ZPRÁV (IEM ASOS / ČERVENEC 2023)
# -------------------------------------------------------------
airports = ["ORD", "ATL", "DFW", "JFK"]
dataframes = []

for airport in airports:
    # Parametry: stanice, rozsah 1. 7. až 31. 7. 2023, čas UTC
    url = (
        f"https://mesonet.agron.iastate.edu/cgi-bin/request/asos.py?"
        f"station={airport}&data=metar&year1=2023&month1=7&day1=1"
        f"&year2=2023&month2=7&day2=31&tz=Etc/UTC&format=onlycomma"
    )

    csv_path = RAW_DIR / f"metar_{airport.lower()}_2023_07.csv"

    # Stažení a uložení do složky data/raw
    response = requests.get(url)
    with open(csv_path, "w", encoding="utf-8") as f:
        f.write(response.text)

    # Načtení do paměti a připojení k celku
    df_temp = pd.read_csv(csv_path)
    dataframes.append(df_temp)
    print(f"K{airport}: staženo {len(df_temp):,} zpráv")

# Sloučení všech 4 stanic do jedné tabulky
df_metar = pd.concat(dataframes, ignore_index=True)

# Kontrola výsledku
print(f"\nCelkem METAR zpráv v tabulce: {len(df_metar):,}")
df_metar.head()

KORD: staženo 9,432 zpráv
KATL: staženo 9,362 zpráv
KDFW: staženo 9,321 zpráv
KJFK: staženo 8,665 zpráv

Celkem METAR zpráv v tabulce: 36,780


,station,valid,metar
0,ORD,2023-07-01 00:00,KORD 010000Z AUTO 26005KT 9SM FEW045 SCT100 29...
1,ORD,2023-07-01 00:05,KORD 010005Z AUTO 28005KT 9SM FEW045 SCT100 29...
2,ORD,2023-07-01 00:10,KORD 010010Z AUTO 27005KT 9SM FEW045 SCT100 29...
3,ORD,2023-07-01 00:15,KORD 010015Z AUTO 26005KT 9SM FEW045 SCT100 29...
4,ORD,2023-07-01 00:20,KORD 010020Z AUTO 25005KT 9SM FEW045 SCT100 29...


V dalším kroku provedeme import leteckých dat a informací o zpoždění (a jeho typu) v červenci 2023.

In [4]:
# -------------------------------------------------------------
# STAŽENÍ A ROZBALENÍ BTS ON-TIME PERFORMANCE (07/2023)
# -------------------------------------------------------------
bts_url = "https://transtats.bts.gov/PREZIP/On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2023_7.zip"
zip_path = RAW_DIR / "bts_2023_07.zip"
csv_path = RAW_DIR / "bts_2023_07.csv"

# 1. Stažení ZIP archivu (cca 30 MB)
print("Stahuji BTS letová data z repozitáře USDOT...")
headers = {"User-Agent": "Mozilla/5.0"}
res = requests.get(bts_url, headers=headers, stream=True, timeout=180)
res.raise_for_status()

with open(zip_path, "wb") as f:
    for chunk in res.iter_content(chunk_size=1024 * 1024):
        if chunk:
            f.write(chunk)

# 2. Rozbalení CSV do složky data/raw a smazání staženého ZIPu
with zipfile.ZipFile(zip_path, "r") as z:
    raw_name = [name for name in z.namelist() if name.endswith(".csv")][0]
    z.extract(raw_name, RAW_DIR)
    os.rename(RAW_DIR / raw_name, csv_path)

zip_path.unlink()
print(f"Hotovo! Rozbaleno do: {csv_path.name}")

# 3. Rychlá kontrola prvních řádků
df_bts_sample = pd.read_csv(csv_path, nrows=3)
print(f"Tabulka má {len(df_bts_sample.columns)} sloupců.")
df_bts_sample.iloc[:, :6]

Stahuji BTS letová data z repozitáře USDOT...
Hotovo! Rozbaleno do: bts_2023_07.csv
Tabulka má 110 sloupců.


,Year,Quarter,Month,DayofMonth,DayOfWeek,FlightDate
0,2023,3,7,2,7,2023-07-02
1,2023,3,7,5,3,2023-07-05
2,2023,3,7,9,7,2023-07-09


Závěrem ještě zjistíme s jak velkými datovými soubory pracujeme.

In [5]:
# -------------------------------------------------------------
# VELIKOST A POČET ŘÁDKŮ SUROVÝCH DAT (PREPARE)
# -------------------------------------------------------------
file_stats = []

for file_path in sorted(RAW_DIR.glob("*.csv")):
    size_mb = file_path.stat().st_size / (1024 * 1024)
    # Rychlý součet řádků bez načítání celého souboru do paměti
    with open(file_path, "rb") as f:
        row_count = sum(1 for _ in f) - 1  # Odečteme hlavičku

    file_stats.append(
        {
            "Soubor": file_path.name,
            "Velikost (MB)": round(size_mb, 2),
            "Počet záznamů": f"{row_count:,}",
        }
    )

df_file_stats = pd.DataFrame(file_stats)
df_file_stats

,Soubor,Velikost (MB),Počet záznamů
0,bts_2023_07.csv,259.93,"601,866"
1,metar_atl_2023_07.csv,0.90,"9,362"
2,metar_dfw_2023_07.csv,0.84,"9,321"
3,metar_jfk_2023_07.csv,0.81,"8,665"
4,metar_ord_2023_07.csv,0.90,"9,432"


## 02 - PROCESS: Čištění dat, příprava METAR zpráv a filtrování letů


* V první části procesu čištění a filtrování relevatních dat nás čeká práce s datovým souborem letů, tj. takzvaného BTS. Tento soubor obsahuje obrovské množství sloupců (110), z nichž je většina pro naší analýzu irelevatních. Pro analýzu zpoždění potřebujeme hlavně data o zpoždění, čísla letadla a důvodu zpoždění, případně letiště vzletu a odletu. První část skriptu tedy filtrujte relevantní sloupce pro naší analýzu.
* Zrušené lety pro nás v této analýze také nejsou důležité, jde nám totiž o zpoždění letů a tak zrušené lety z našeho souboru vyřadíme.
* Lety bez registrace (tzv. tail number) jsou pro naší analýzu bezpředmětné, nemůžeme je totiž spárovat s dalšími linkami.
* U nezpožděných letů nahradíme údaj nA dobou zpoždění 0.


In [6]:
# Cesta k surovému souboru z BTS
RAW_DIR = Path("data/raw")
bts_file = RAW_DIR / "bts_2023_07.csv"

# 1. Výběr pouze relevantních sloupců pro analýzu zpoždění a rotací
columns_to_keep = [
    "FlightDate", "Reporting_Airline", "Tail_Number", "Flight_Number_Reporting_Airline",
    "Origin", "Dest", "CRSDepTime", "DepTime", "DepDelay",
    "CRSArrTime", "ArrTime", "ArrDelay", "Cancelled",
    "CarrierDelay", "WeatherDelay", "NASDelay", "LateAircraftDelay"
]

print("Načítám vybrané sloupce z BTS dat...")
df_flights = pd.read_csv(bts_file, usecols=columns_to_keep, low_memory=False)
initial_rows = len(df_flights)
print(f"Původní počet záznamů za celé USA: {initial_rows:,}")

# 2. Odstranění zrušených letů (Cancelled == 1)
# Zrušené lety neproběhly, nemají reálné letové časy a netvoří rotaci stroje
df_flights = df_flights[df_flights["Cancelled"] == 0].copy()
cancelled_diff = initial_rows - len(df_flights)
print(f"Odstraněno zrušených letů: {cancelled_diff:,}")

# 3. Odstranění letů bez registrace letadla (Tail_Number is NaN)
# Bez imatrikulace není možné let zařadit do řetězce rotace konkrétního stroje
df_flights = df_flights.dropna(subset=["Tail_Number"]).copy()

# 4. Ošetření chybějících hodnot v kategoriích zpoždění
# BTS vykazuje NaN, pokud zpoždění nevzniklo nebo bylo pod 15 minut; nahradíme nulou
delay_cols = ["CarrierDelay", "WeatherDelay", "NASDelay", "LateAircraftDelay"]
df_flights[delay_cols] = df_flights[delay_cols].fillna(0)

print(f"Konečný počet platných záznamů po základní očistě: {len(df_flights):,}")
df_flights.head(3)

Načítám vybrané sloupce z BTS dat...
Původní počet záznamů za celé USA: 601,866
Odstraněno zrušených letů: 14,606
Konečný počet platných záznamů po základní očistě: 587,260


,FlightDate,Reporting_Airline,Tail_Number,Flight_Number_Reporting_Airline,Origin,Dest,CRSDepTime,DepTime,DepDelay,CRSArrTime,ArrTime,ArrDelay,Cancelled,CarrierDelay,WeatherDelay,NASDelay,LateAircraftDelay
0,2023-07-02,9E,N307PQ,4900,DTW,DSM,946,942.0,-4.0,1034,1015.0,-19.0,0.0,0.0,0.0,0.0,0.0
1,2023-07-05,9E,N232PQ,4900,DTW,DSM,1355,1356.0,1.0,1442,1431.0,-11.0,0.0,0.0,0.0,0.0,0.0
2,2023-07-09,9E,N480PX,4900,DTW,DSM,946,942.0,-4.0,1034,1018.0,-16.0,0.0,0.0,0.0,0.0,0.0


Pro výpočet doby zpoždění a klíčových metrik ohledně zpoždění jsou důležité časy odletu a příletu. Protože se v USA používají různá časová pásma, využijeme tohoto skriptu ke sjednocení časových údajů do standardního formátu používaného v letectví, tzv. UTC času.

In [7]:
# -------------------------------------------------------------
# KROK 2: NORMALIZACE ČASŮ A PŘEVOD DO UTC
# -------------------------------------------------------------

# 1. Pomocná funkce: převod float/int typu 905.0 na řetězec '09:05'
def bts_time_to_str(val):
    if pd.isna(val):
        return None
    val_int = int(val)
    # BTS uvádí půlnoc jako 2400 -> normalizujeme na 0000
    if val_int == 2400:
        val_int = 0
    s = str(val_int).zfill(4)
    return f"{s[:2]}:{s[2:]}"

print("Normalizuji časové formáty...")
for col in ["CRSDepTime", "DepTime", "CRSArrTime", "ArrTime"]:
    df_flights[f"{col}_str"] = df_flights[col].apply(bts_time_to_str)

# 2. Vytvoření lokálního datetime pro plánovaný odlet (CRSDepTime)
# Spojíme FlightDate (YYYY-MM-DD) a vyčištěný čas (HH:MM)
df_flights["CRSDep_dt_local"] = pd.to_datetime(
    df_flights["FlightDate"] + " " + df_flights["CRSDepTime_str"],
    errors="coerce"
)

# 3. Namapování časových zón hubů (v červenci platí letní čas EDT / CDT)
tz_map = {
    "ORD": "America/Chicago",  # CDT (UTC-5)
    "DFW": "America/Chicago",  # CDT (UTC-5)
    "ATL": "America/New_York", # EDT (UTC-4)
    "JFK": "America/New_York"  # EDT (UTC-4)
}

# 4. Převod odletového času do UTC pro naše sledované huby
def convert_to_utc(row):
    origin = row["Origin"]
    dt_local = row["CRSDep_dt_local"]
    if pd.isna(dt_local) or origin not in tz_map:
        return pd.NaT
    # Lokalizace do časového pásma letiště a konverze do UTC
    return dt_local.tz_localize(tz_map[origin]).tz_convert("UTC")

print("Převádím plánované odlety na hubu do UTC...")
hub_mask = df_flights["Origin"].isin(tz_map.keys())
df_flights.loc[hub_mask, "CRSDep_dt_utc"] = df_flights[hub_mask].apply(convert_to_utc, axis=1)

# Kontrola výsledku
sample_cols = ["FlightDate", "Origin", "CRSDepTime", "CRSDepTime_str", "CRSDep_dt_utc"]
df_flights.loc[hub_mask, sample_cols].head(5)

Normalizuji časové formáty...
Převádím plánované odlety na hubu do UTC...


,FlightDate,Origin,CRSDepTime,CRSDepTime_str,CRSDep_dt_utc
51,2023-07-02,JFK,1626,16:26,2023-07-02 20:26:00+00:00
52,2023-07-03,JFK,1626,16:26,2023-07-03 20:26:00+00:00
53,2023-07-04,JFK,1626,16:26,2023-07-04 20:26:00+00:00
54,2023-07-05,JFK,1626,16:26,2023-07-05 20:26:00+00:00
55,2023-07-06,JFK,1626,16:26,2023-07-06 20:26:00+00:00


Po vyčištění a relevantního filtrování prvního datového souboru (BTS) nás nyní čeká ještě práce s druhým datovým souborem - METARY (tj. zprávami o počasí).

Prvním úkolem je odstranění případných duplicitních METARů, které nepotřebujeme. Dalším krokem v práci s METARy je  extrakce pro nás relevatních informací - pro potřeby analýzy například nepotřebujeme údaje o teplotě nebo rosném bodu.

Cílem je převést surový textový řetězec na parametry: dohlednost v mílích (SM), strop oblačnosti ve stopách (Ceiling), bouřkové jevy (TSRA) a určit oficiální FAA kategorii:
* LIFR (zhoršené podmínky létání dle přístrojů)
* IFR (podmínky létání dle přístrojů)
* MVFR (zhoršené podmínky létání dle vidu)
* VFR (podmínky létání dle vidu)

In [8]:
# -------------------------------------------------------------
# KROK 3: REGEX PARSING METAR ZPRÁV A LETOVÉ KATEGORIE
# -------------------------------------------------------------
import re

# Odstranění duplicitních zpráv pro stejnou stanici a časové razítko
initial_count = len(df_metar)
df_metar = df_metar.drop_duplicates(subset=["station", "valid"]).copy()
print(f"Odstraněno duplicit: {initial_count - len(df_metar):,}")


def parse_metar_record(raw_metar):
    """Parsuje surový řetězec METARu a vrací dohlednost (SM), ceiling (ft),

    indikátor bouřky a vypočtenou letovou kategorii.
    """
    if not isinstance(raw_metar, str) or not raw_metar.strip():
        return pd.Series([None, None, False, "UNKNOWN"])

    # 1. Extrakce dohlednosti (např. '10SM', '1/2SM', '2 1/2SM')
    vis_match = re.search(r"\b((\d+)\s+)?(\d+/\d+|\d+)\s*SM\b", raw_metar)
    vis = None
    if vis_match:
        whole = vis_match.group(2)
        frac_or_int = vis_match.group(3)
        if "/" in frac_or_int:
            num, denom = frac_or_int.split("/")
            val = float(num) / float(denom)
            vis = float(whole) + val if whole else val
        else:
            vis = float(frac_or_int)

    # 2. Extrakce stropu oblačnosti (nejnižší vrstva BKN nebo OVC)
    # Hledá BKNxxx nebo OVCxxx, hodnota je ve stovkách stop
    cloud_layers = re.findall(r"\b(BKN|OVC)(\d{3})\b", raw_metar)
    ceiling = None
    if cloud_layers:
        heights = [int(layer[1]) * 100 for layer in cloud_layers]
        ceiling = min(heights)

    # 3. Detekce bouřkových jevů (TS, TSRA, VCTS, mraky CB)
    storm = bool(re.search(r"\b(\+|-)?(TS|TSRA|VCTS)\b|\bCB\b", raw_metar))

    # 4. Stanovení oficiální letové kategorie (FAA)
    # Výchozí stav: pokud není BKN/OVC, ceiling je neomezený
    eff_ceiling = ceiling if ceiling is not None else 99999
    eff_vis = vis if vis is not None else 10.0

    if eff_vis < 1.0 or eff_ceiling < 500:
        cat = "LIFR"
    elif eff_vis < 3.0 or eff_ceiling < 1000:
        cat = "IFR"
    elif eff_vis <= 5.0 or eff_ceiling <= 3000:
        cat = "MVFR"
    else:
        cat = "VFR"

    return pd.Series([vis, ceiling, storm, cat])


print("Parsuji METAR data...")
df_metar[["visibility_sm", "ceiling_ft", "has_storm", "flight_category"]] = (
    df_metar["metar"].apply(parse_metar_record)
)

# Převedení časového razítka METARu na UTC datetime a seřazení
df_metar["valid_utc"] = pd.to_datetime(
    df_metar["valid"], errors="coerce"
).dt.tz_localize(None)

# Zobrazení zastoupení kategorií
print("\nRozložení letových kategorií na hubech (07/2023):")
print(df_metar["flight_category"].value_counts())
df_metar[
    [
        "station",
        "valid",
        "visibility_sm",
        "ceiling_ft",
        "has_storm",
        "flight_category",
    ]
].head(5)

Odstraněno duplicit: 84
Parsuji METAR data...

Rozložení letových kategorií na hubech (07/2023):
flight_category
VFR     33043
MVFR     2678
IFR       951
LIFR       24
Name: count, dtype: int64


,station,valid,visibility_sm,ceiling_ft,has_storm,flight_category
0,ORD,2023-07-01 00:00,9.0,NaN,False,VFR
1,ORD,2023-07-01 00:05,9.0,NaN,False,VFR
2,ORD,2023-07-01 00:10,9.0,NaN,False,VFR
3,ORD,2023-07-01 00:15,9.0,NaN,False,VFR
4,ORD,2023-07-01 00:20,9.0,NaN,False,VFR


Nyní nás čeká poslední krok této přípravné fáze, a to spojení METAR zpráv s příslušnými lety.

In [9]:
# -------------------------------------------------------------
# KROK 4: PÁROVÁNÍ LETŮ S METEOROLOGIÍ (MERGE_ASOF)
# -------------------------------------------------------------

target_hubs = ["ORD", "ATL", "DFW", "JFK"]

# 1. Filtrace letů z vybraných hubů a sjednocení typu času (tz-naive)
df_hub_departures = df_flights[
    df_flights["Origin"].isin(target_hubs)
    & df_flights["CRSDep_dt_utc"].notna()
].copy()

# Odstraníme explicitní timezone flag pro shodu s METAR daty
df_hub_departures["CRSDep_dt_utc"] = df_hub_departures[
    "CRSDep_dt_utc"
].dt.tz_localize(None)
df_hub_departures = df_hub_departures.sort_values(
    by="CRSDep_dt_utc"
).reset_index(drop=True)

# 2. Příprava METAR zpráv pro spojení
metar_cols = [
    "station",
    "valid_utc",
    "flight_category",
    "has_storm",
    "visibility_sm",
    "ceiling_ft",
]
df_metar_clean = (
    df_metar[metar_cols]
    .dropna(subset=["valid_utc"])
    .sort_values(by="valid_utc")
    .reset_index(drop=True)
)

print(
    f"Páruji {len(df_hub_departures):,} odletů z hubů s METAR hlášeními (tolerance 75 min)..."
)

# 3. As-of merge: přiřazení nejbližšího předchozího METARu na daném letišti
df_matched = pd.merge_asof(
    df_hub_departures,
    df_metar_clean,
    left_on="CRSDep_dt_utc",
    right_on="valid_utc",
    left_by="Origin",
    right_by="station",
    direction="backward",
    tolerance=pd.Timedelta("75m"),
)

# 4. Kontrola úspěšnosti spárování
matched_count = df_matched["flight_category"].notna().sum()
match_pct = (matched_count / len(df_matched)) * 100
print(f"Úspěšně přiřazeno počasí: {matched_count:,} letů ({match_pct:.2f} %)")

# Ukázka spárovaných dat
preview_cols = [
    "FlightDate",
    "Reporting_Airline",
    "Tail_Number",
    "Origin",
    "Dest",
    "CRSDep_dt_utc",
    "DepDelay",
    "flight_category",
    "has_storm",
    "LateAircraftDelay",
    "WeatherDelay",
]
df_matched[preview_cols].head(5)

Páruji 87,655 odletů z hubů s METAR hlášeními (tolerance 75 min)...
Úspěšně přiřazeno počasí: 84,232 letů (96.09 %)


,FlightDate,Reporting_Airline,Tail_Number,Origin,Dest,CRSDep_dt_utc,DepDelay,flight_category,has_storm,LateAircraftDelay,WeatherDelay
0,2023-07-01,AS,N478AS,ORD,ANC,2023-07-01 05:55:00,26.0,VFR,False,0.0,0.0
1,2023-07-01,B6,N973JT,JFK,SJU,2023-07-01 09:07:00,-8.0,VFR,False,0.0,0.0
2,2023-07-01,AA,N9013A,ATL,CLT,2023-07-01 09:10:00,-4.0,VFR,False,0.0,0.0
3,2023-07-01,NK,N611NK,ATL,FLL,2023-07-01 09:20:00,287.0,VFR,False,0.0,0.0
4,2023-07-01,UA,N844UA,ATL,IAD,2023-07-01 09:45:00,0.0,VFR,False,0.0,0.0


Závěrečný kód pro převod vyčištěných a spárovaných dat do nové .csv tabulky, kterou využijeme v analýze v další fázi.

In [10]:
# -------------------------------------------------------------
# KROK 5: INTEGRITA A EXPORT DO DATA/PROCESSED
# -------------------------------------------------------------
from pathlib import Path

PROCESSED_DIR = Path("data/processed")
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
output_path = PROCESSED_DIR / "clean_flights_weather.csv"

# Ponecháme pouze úspěšně spárované lety
df_final = df_matched.dropna(subset=["flight_category"]).copy()

# Uložení do CSV
print(f"Ukládám {len(df_final):,} vyčištěných a spárovaných letů do {output_path}...")
df_final.to_csv(output_path, index=False)

print("Export dokončen. Fáze PROCESS je kompletní!")
print(f"Velikost výsledného datasetu: {df_final.shape[0]:,} řádků a {df_final.shape[1]} sloupců.")

Ukládám 84,232 vyčištěných a spárovaných letů do data/processed/clean_flights_weather.csv...
Export dokončen. Fáze PROCESS je kompletní!
Velikost výsledného datasetu: 84,232 řádků a 29 sloupců.


In [11]:
# -------------------------------------------------------------
# NÁHLEDOVÁ KONTROLA NÁHODNÝCH ZÁZNAMŮ
# -------------------------------------------------------------
audit_cols = [
    "FlightDate",
    "Reporting_Airline",
    "Tail_Number",
    "Origin",
    "Dest",
    "CRSDep_dt_utc",
    "flight_category",
    "has_storm",
    "visibility_sm",
    "ceiling_ft",
    "DepDelay",
    "LateAircraftDelay",
    "NASDelay",
    "WeatherDelay",
]

# Výběr 5 náhodných řádků (random_state zaručuje reprodukovatelnost při opakovaném běhu)
df_sample = df_final[audit_cols].sample(n=5, random_state=42)
df_sample

,FlightDate,Reporting_Airline,Tail_Number,Origin,Dest,CRSDep_dt_utc,flight_category,has_storm,visibility_sm,ceiling_ft,DepDelay,LateAircraftDelay,NASDelay,WeatherDelay
49694,2023-07-18,OO,N762SK,DFW,SWO,2023-07-19 02:16:00,VFR,False,10.0,NaN,-7.0,0.0,0.0,0.0
78272,2023-07-28,UA,N18223,ORD,LGA,2023-07-28 22:50:00,VFR,False,10.0,3800.0,104.0,0.0,117.0,0.0
66340,2023-07-24,B6,N583JB,JFK,MIA,2023-07-24 21:10:00,VFR,False,10.0,NaN,151.0,109.0,0.0,0.0
41381,2023-07-16,AS,N402AS,DFW,SEA,2023-07-16 11:00:00,VFR,False,10.0,NaN,53.0,28.0,0.0,0.0
26106,2023-07-10,DL,N881DN,ATL,MEM,2023-07-10 22:38:00,VFR,False,10.0,NaN,-3.0,0.0,0.0,0.0


## 03 - ANALYSE: ANALÝZA DAT DLE ZADANÝCH PARAMETRŮ


V této části nás čeká analýza dat. Začneme výpočtem potřebných metrik ohledně počasí. Konkrétně v následujícím kódu:


*   Vypočítáme celkový objem letů, průměrné zpoždění a medián zpoždění v závislosti na meteorologických podmínkách a klasifikaci FFA (VFR, IFR, ...).
*  Zjistíme počet letů se zpožděním v dané kategorii.
* Analyzujeme, jaký vliv má na zpoždění konvektivní počasí (bouřky, mraky typu cumulonimbus apod.).
* Podíváme se na poměrové srovnání důvodů zpoždění WeatherDelay vs. LateAircraftDelay.



In [13]:
# 1. Načtení dat
file_path = Path("data/processed/clean_flights_weather.csv")
df = pd.read_csv(file_path, parse_dates=["CRSDep_dt_utc"])

# 2. Pomocné sloupce
df["is_delayed_15"] = df["DepDelay"] > 15
cat_order = ["VFR", "MVFR", "IFR", "LIFR"]
df["flight_category"] = pd.Categorical(
    df["flight_category"], categories=cat_order, ordered=True
)


def get_delay_summary(group_col):
  summary = (
      df.groupby(group_col, observed=True)
      .agg(
          total_flights=("DepDelay", "count"),
          pct_delayed_15=("is_delayed_15", lambda x: np.mean(x) * 100),
          avg_dep_delay=("DepDelay", "mean"),
          median_dep_delay=("DepDelay", "median"),
          sum_weather_delay=("WeatherDelay", "sum"),
          sum_late_aircraft=("LateAircraftDelay", "sum"),
      )
      .reset_index()
  )
  # Přímý výpočet poměru Late Aircraft ku Weather zpoždění
  summary["late_to_weather_ratio"] = (
      summary["sum_late_aircraft"] / summary["sum_weather_delay"].replace(0, np.nan)
  )
  return summary


# 3. Výpis kategorií
cat_summary = get_delay_summary("flight_category")
print("=== METRIKY DLE LETOVÝCH KATEGORIÍ FAA ===")
print(
    cat_summary.to_string(
        index=False,
        formatters={
            "total_flights": "{:,}".format,
            "pct_delayed_15": "{:.1f} %".format,
            "avg_dep_delay": "{:.1f} min".format,
            "median_dep_delay": "{:.0f} min".format,
            "sum_weather_delay": "{:,.0f} min".format,
            "sum_late_aircraft": "{:,.0f} min".format,
            "late_to_weather_ratio": "{:.1f}x".format,
        },
    )
)

# 4. Celkový poměr za celý auditovaný vzorek
total_weather = df["WeatherDelay"].sum()
total_late = df["LateAircraftDelay"].sum()
overall_ratio = total_late / total_weather if total_weather > 0 else 0

print(f"\n=== CELKOVÝ POMĚR ZPOŽDĚNÍ (VŠECHNY PODMÍNKY) ===")
print(f"Oficiální Weather Delay:      {total_weather:,.0f} min")
print(f"Vykázané Late Aircraft Delay: {total_late:,.0f} min")
print(
    f"Poměr (Late Aircraft / Weather): {overall_ratio:.2f}x (na 1 minutu oficiálního počasí připadá {overall_ratio:.1f} minut čekání na předchozí letadlo)"
)

=== METRIKY DLE LETOVÝCH KATEGORIÍ FAA ===
flight_category total_flights pct_delayed_15 avg_dep_delay median_dep_delay sum_weather_delay sum_late_aircraft late_to_weather_ratio
            VFR        77,597         30.3 %      23.0 min            0 min       100,841 min       742,583 min                  7.4x
           MVFR         5,170         30.9 %      26.1 min            0 min        12,474 min        50,036 min                  4.0x
            IFR         1,439         40.0 %      30.8 min            7 min         5,997 min        15,500 min                  2.6x
           LIFR            26         57.7 %      74.6 min           28 min           126 min           391 min                  3.1x

=== CELKOVÝ POMĚR ZPOŽDĚNÍ (VŠECHNY PODMÍNKY) ===
Oficiální Weather Delay:      119,438 min
Vykázané Late Aircraft Delay: 808,510 min
Poměr (Late Aircraft / Weather): 6.77x (na 1 minutu oficiálního počasí připadá 6.8 minut čekání na předchozí letadlo)


### Interpretace 1. části analýzy
* **Rozdělení dle FAA kategorií:** Při přechodu z optimálního VFR do nepříznivých podmínek IFR a LIFR roste podíl zpožděných letů (> 15 min) z 30,3 % na 40,0 % (IFR) až 57,7 % (LIFR). Průměrné zpoždění u LIFR dosahuje 74,6 minuty oproti 23,0 min u VFR.
* **Poměr Late Aircraft vs. Weather (6,8 : 1):** Vykázané zpoždění z čekání na letadlo (`LateAircraftDelay`: 808 510 min) převyšuje oficiální meteorologické zpoždění (`WeatherDelay`: 119 438 min) téměř sedminásobně.
* **Skrytá meteorologická stopa ve VFR:** I při optimálním VFR počasí bylo vykázáno přes 742 tisíc minut zpoždění *Late Aircraft*, což indikuje masivní přelévání ranních zpoždění z exponovaných uzlů do zbytku letového řádu.

Další část analýzy je věnována tzv. ,,řetězení letů". Letecké společnosti během dne samozřejmě používají letadlo víckrát, dost často je letecký řád velmi napjatý a každé zpoždění indukuje zpoždění navazujících letadel. V této části nás čekají následující úkoly:

* Seřazení letů podle registrace letadla (`Tail_Number`) a plánovaného odletu v UTC (`CRSDep_dt_utc`).
* Identifikace sekvence letů (číslo legu) konkrétního stroje v rámci jednoho provozního dne.
* Kalkulace plánovaného rozestupu mezi lety (turnaround time).

In [14]:
# ==============================================================================
# KROK 2: REKONSTRUKCE ROTACÍ LETADEL (TAIL NUMBER TRACKING)
# ==============================================================================

# 1. Úkol: Seřazení podle registrace letadla a plánovaného odletu v UTC
df = df.sort_values(by=["Tail_Number", "CRSDep_dt_utc"]).reset_index(drop=True)

# 2. Úkol: Sekvence letů konkrétního stroje v rámci jednoho dne (leg 1, 2, 3...)
df["leg_sequence"] = df.groupby(["Tail_Number", "FlightDate"]).cumcount() + 1

# 3. Úkol: Plánovaný rozestup (turnaround buffer) mezi odlety stejného stroje v daném dni
# Posuneme čas předchozího odletu o 1 řádek v rámci stejného letadla
df["prev_crs_dep_utc"] = df.groupby("Tail_Number")["CRSDep_dt_utc"].shift(1)
df["prev_flight_date"] = df.groupby("Tail_Number")["FlightDate"].shift(1)

# Spočítáme rozdíl v minutách mezi odlety
dep_diff_minutes = (
    df["CRSDep_dt_utc"] - df["prev_crs_dep_utc"]
).dt.total_seconds() / 60.0

# Buffer přiřadíme jen návazným letům ve stejném dni (ne prvnímu letu dne)
df["turnaround_buffer_min"] = np.where(
    (df["leg_sequence"] > 1) & (df["FlightDate"] == df["prev_flight_date"]),
    dep_diff_minutes,
    np.nan,
)

# Úklid pomocných sloupců
df = df.drop(columns=["prev_crs_dep_utc", "prev_flight_date"])

# ==============================================================================
# KONTROLNÍ NÁHLED VÝSLEDKŮ
# ==============================================================================
print("=== SOUHRN ROTACÍ LETADEL ===")
print(f"Unikátních letadel v datasetu: {df['Tail_Number'].nunique():,}")
print(f"Maximální počet odletů jednoho stroje z hubů za den: {df['leg_sequence'].max()}")
print(
    f"Průměrný rozestup mezi odlety stejného stroje: {df['turnaround_buffer_min'].dropna().mean():.1f} min"
)

# Ukázka konkrétní rotace stroje se 3 a více lety
sample_tail = df[df["leg_sequence"] >= 3]["Tail_Number"].iloc[0]
sample_date = df[df["Tail_Number"] == sample_tail]["FlightDate"].iloc[0]

cols_to_show = [
    "Tail_Number",
    "FlightDate",
    "leg_sequence",
    "Origin",
    "Dest",
    "CRSDep_dt_utc",
    "DepDelay",
    "turnaround_buffer_min",
    "LateAircraftDelay",
    "WeatherDelay",
]

sample_rotation = df[
    (df["Tail_Number"] == sample_tail) & (df["FlightDate"] == sample_date)
][cols_to_show]

print(f"\n=== UKÁZKA ROTACE LETADLA {sample_tail} DNE {sample_date} ===")
print(sample_rotation.to_string(index=False))

=== SOUHRN ROTACÍ LETADEL ===
Unikátních letadel v datasetu: 4,817
Maximální počet odletů jednoho stroje z hubů za den: 6
Průměrný rozestup mezi odlety stejného stroje: 370.5 min

=== UKÁZKA ROTACE LETADLA N101DQ DNE 2023-07-01 ===
Tail_Number FlightDate  leg_sequence Origin Dest       CRSDep_dt_utc  DepDelay  turnaround_buffer_min  LateAircraftDelay  WeatherDelay
     N101DQ 2023-07-01             1    ATL  LGA 2023-07-01 18:25:00      -2.0                    NaN                0.0           0.0


In [15]:
# Vybereme první letadlo, které má v jednom dni alespoň 3 odlety z hubů
multi_leg_tail = df[df["leg_sequence"] >= 3]["Tail_Number"].iloc[0]
multi_leg_date = df[
    (df["Tail_Number"] == multi_leg_tail) & (df["leg_sequence"] >= 3)
]["FlightDate"].iloc[0]

# Zobrazíme jeho celodenní rotaci
cols_preview = [
    "Tail_Number",
    "FlightDate",
    "leg_sequence",
    "Origin",
    "Dest",
    "CRSDep_dt_utc",
    "DepDelay",
    "turnaround_buffer_min",
    "LateAircraftDelay",
    "WeatherDelay",
]
df[(df["Tail_Number"] == multi_leg_tail) & (df["FlightDate"] == multi_leg_date)][
    cols_preview
]

,Tail_Number,FlightDate,leg_sequence,Origin,Dest,CRSDep_dt_utc,DepDelay,turnaround_buffer_min,LateAircraftDelay,WeatherDelay
1,N101DQ,2023-07-02,1,ATL,IAH,2023-07-02 11:05:00,0.0,NaN,0.0,0.0
2,N101DQ,2023-07-02,2,ATL,LGA,2023-07-02 17:30:00,17.0,385.0,0.0,0.0
3,N101DQ,2023-07-02,3,ATL,SAN,2023-07-03 01:16:00,-1.0,466.0,0.0,0.0


### Interpretace 2. části analýzy:
* **Rozsah sledované flotily:** V datasetu bylo identifikováno **4 817 unikátních letadel** (`Tail_Number`). Maximální denní počet odletů jednoho stroje z auditovaných hubů dosáhl **6 legů**, což definuje maximální teoretickou hloubku propagace primárního zpoždění.
* **Provozní rozestupy (`turnaround_buffer_min`):** Průměrný interval mezi dvěma odlety téhož letadla z hubů činí **370,5 minuty (~6,2 hodiny)**. Tato hodnota odráží fakt, že letadlo mezi dvěma hubovými odlety často absolvuje rotaci na mimohubové letiště (outstation) a zpět.
* **Důkaz absorpce na reálném příkladu (letoun N101DQ, 2. 7. 2023):**
  * **Leg 1 (ATL $\rightarrow$ IAH):** Odlet v 11:05 UTC načas (`DepDelay = 0 min`).
  * **Leg 2 (ATL $\rightarrow$ LGA):** Odlet v 17:30 UTC se zpožděním **17 minut** (překročení hranice On-Time).
  * **Leg 3 (ATL $\rightarrow$ SAN):** Noční odlet v 01:16 UTC následujícího dne; díky bufferu **466 minut** letadlo zpoždění zcela vstřebalo a odletělo s předstihem **-1 minuta**.
* **Význam pro další krok:** Byla úspěšně vytvořena kostra datového souboru, kterou využijeme v dalším kroku. Propojení letů přes `Tail_Number` a `leg_sequence` umožňuje v Kroku 3 zahájit trasování dominového efektu a aplikovat absorpční cutoff.

V této části navážeme na předchozí práci, kdy jsme sledovali řetězení letů. Budeme nyní zkoumat vliv řetězení a zpoždění. Čekají nás tyto úkoly:

* Primární meteorologický spouštěč: Let se zpožděním `DepDelay > 15 min` v podmínkách IMC (`MVFR`, `IFR`, `LIFR`) nebo při bouřce (`has_storm == True`).
* Sekundární indukované zpoždění: Navazující lety stejného stroje (`Tail_Number`) se zpožděním `DepDelay > 15 min` a vykázaným `LateAircraftDelay > 0`.
* Absorpční cutoff: Řetězec zaniká, jakmile zpoždění klesne na $\le 15$ minut nebo s koncem provozního dne (nové datum).
* Kaskádový index (Cascade Multiplier): Poměr celkových indukovaných sekundárních minut ku primárním meteorologickým minutám.

In [16]:
# ==============================================================================
# KROK 3: DETEKCE SPOUŠTĚČŮ, TRASOVÁNÍ KASKÁD A KASKÁDOVÝ INDEX
# ==============================================================================

# 1. Definice primárního meteorologického spouštěče
# Podmínka: zpoždění > 15 min A ZÁROVEŇ nepříznivé počasí (IMC nebo bouřka)
is_bad_weather = (df["flight_category"].isin(["MVFR", "IFR", "LIFR"])) | (
    df["has_storm"] == True
)
df["is_primary_weather_trigger"] = (df["DepDelay"] > 15) & is_bad_weather

# 2. Trasování kaskády v chronologické rotaci letadla
is_cascade_leg = []
induced_delay_minutes = []

active_cascade = False
current_tail = None
current_date = None

for row in df.itertuples():
  # Noční cutoff / změna letadla: nový den nebo jiné letadlo kaskádu ukončí
  if row.Tail_Number != current_tail or row.FlightDate != current_date:
    active_cascade = False
    current_tail = row.Tail_Number
    current_date = row.FlightDate

  # A. Primární spouštěč aktivuje dominový efekt pro další lety
  if row.is_primary_weather_trigger:
    active_cascade = True
    is_cascade_leg.append(False)  # Primární let není sekundární
    induced_delay_minutes.append(0.0)

  # B. Navazující let v běžící kaskádě
  elif active_cascade:
    # Pokud letadlo stále nese zpoždění a vykazuje čekání na stroj
    if row.DepDelay > 15 and row.LateAircraftDelay > 0:
      is_cascade_leg.append(True)
      # Připisujeme skutečně vykázané LateAircraftDelay (maximálně do výše DepDelay)
      induced_delay_minutes.append(min(row.LateAircraftDelay, row.DepDelay))
    else:
      # Absorpční cutoff: zpoždění vstřebáno (<= 15 min) -> řetězec končí
      active_cascade = False
      is_cascade_leg.append(False)
      induced_delay_minutes.append(0.0)
  else:
    is_cascade_leg.append(False)
    induced_delay_minutes.append(0.0)

df["is_induced_delay"] = is_cascade_leg
df["induced_delay_min"] = induced_delay_minutes

# ==============================================================================
# 3. VÝPOČET KASKÁDOVÉHO INDEXU A BILANCE
# ==============================================================================
# Primární minuty na spouštěčích vs. indukované sekundární minuty
primary_weather_min = df.loc[
    df["is_primary_weather_trigger"], "DepDelay"
].sum()
secondary_induced_min = df["induced_delay_min"].sum()

cascade_multiplier = (
    secondary_induced_min / primary_weather_min
    if primary_weather_min > 0
    else 0.0
)

# Počty zasažených letů
primary_count = df["is_primary_weather_trigger"].sum()
induced_count = df["is_induced_delay"].sum()
flight_multiplier = (
    induced_count / primary_count if primary_count > 0 else 0.0
)

print("=== BILANCE KASKÁDOVÉHO EFEKTU ===")
print(f"Primární spouštěcí lety:               {primary_count:,}")
print(f"Následně indukované zpožděné lety:      {induced_count:,}")
print(f"Index zasažených letů:                  {flight_multiplier:.2f}x")

print(
    f"\nPrimární meteorologické zpoždění:       {primary_weather_min:,.0f} min"
)
print(
    f"Indukované sekundární zpoždění:         {secondary_induced_min:,.0f} min"
)
print(
    f"Kaskádový index (Cascade Multiplier):   {cascade_multiplier:.2f}x (na 1 minutu počasí připadá {cascade_multiplier:.2f} minut indukovaného zpoždění)"
)

=== BILANCE KASKÁDOVÉHO EFEKTU ===
Primární spouštěcí lety:               3,917
Následně indukované zpožděné lety:      529
Index zasažených letů:                  0.14x

Primární meteorologické zpoždění:       344,091 min
Indukované sekundární zpoždění:         38,317 min
Kaskádový index (Cascade Multiplier):   0.11x (na 1 minutu počasí připadá 0.11 minut indukovaného zpoždění)


### Interpretace 3. části analýzy:
* **Detekované primární spouštěče:** Identifikováno celkem **3 917 letů**, které odletěly se zpožděním > 15 min přímo v nepříznivých meteorologických podmínkách (IMC nebo bouřka), s kumulovaným primárním zpožděním **344 091 minut** (~88 min/let).
* **Indukovaná sekundární zpoždění (Hub-to-Hub):** Na navazujících segmentech stejných letadel v tentýž den bylo bezpečně vystopováno **529 sekundárně zasažených letů** a **38 317 minut** zpoždění vykázaného jako `LateAircraftDelay`.
* **Kaskádový index ($0{,}11\times$):** Na každou 1 minutu primárního meteorologického zpoždění na hubu připadá **0,11 minuty** prokazatelného sekundárního zpoždění na dalším hubovém odletu.
* **Metodické ohraničení (Konzervativní dolní mez):** Model sleduje rotace výhradně mezi 4 hlavními uzly (ORD, ATL, DFW, JFK). Mezilehlé lety na regionální letiště (outstations) zůstávají mimo záběr a dlouhý průměrný rozestup mezi hubovými odlety (~6,2 h) poskytuje prostor pro provozní absorpci skluzu. Zjištěných 38,3 tis. minut tak představuje garantovanou spodní hranici dominového efektu.

Posledním úkolem samotné analýzy je porovnání daných letišť (hubů) a též kritickkých dnů:
* Srovnání odolnosti hubů: Analýza zranitelnosti uzlů ORD, ATL, DFW a JFK vůči generování a šíření kaskádových zpoždění.
* Finanční vyčíslení (FAA Benchmark): Kalkulace celkových škod pomocí přímých provozních nákladů ve výši **100 USD za minutu zpoždění**.
* Denní dynamika šíření: Identifikace rizikových časových oken (UTC hodiny) s nejvyšší kumulací indukovaného zpoždění.

In [17]:
# ==============================================================================
# KROK 4: AUDIT ODOLNOSTI HUBŮ A FINANČNÍ DOPADY
# ==============================================================================

# Standardní provozní náklad dle FAA / A4A benchmarku (100 USD / minuta)
COST_PER_MINUTE_USD = 100

# 1. Agregace metrik podle hubu (Origin)
hub_audit = (
    df.groupby("Origin")
    .agg(
        total_flights=("DepDelay", "count"),
        primary_triggers=("is_primary_weather_trigger", "sum"),
        induced_flights=("is_induced_delay", "sum"),
        primary_delay_min=("DepDelay", lambda x: x[df.loc[x.index, "is_primary_weather_trigger"]].sum()),
        induced_delay_min=("induced_delay_min", "sum"),
        total_dep_delay=("DepDelay", "sum"),
        official_weather_delay=("WeatherDelay", "sum"),
        official_late_aircraft_delay=("LateAircraftDelay", "sum"),
    )
    .reset_index()
)

# Dopočet kaskádového indexu a finančních škod pro jednotlivé huby
hub_audit["cascade_multiplier"] = (
    hub_audit["induced_delay_min"] / hub_audit["primary_delay_min"]
).round(3)

hub_audit["primary_cost_usd"] = (
    hub_audit["primary_delay_min"] * COST_PER_MINUTE_USD
)
hub_audit["induced_cost_usd"] = (
    hub_audit["induced_delay_min"] * COST_PER_MINUTE_USD
)

# Seřazení podle objemu indukovaného zpoždění (nejzranitelnější uzly nahoře)
hub_audit = hub_audit.sort_values(by="induced_delay_min", ascending=False).reset_index(drop=True)

print("=== SROVNÁNÍ ODOLNOSTI HUBŮ (KORD vs. KATL vs. KDFW vs. KJFK) ===")
cols_hub_view = [
    "Origin",
    "total_flights",
    "primary_triggers",
    "induced_flights",
    "cascade_multiplier",
    "primary_delay_min",
    "induced_delay_min",
    "induced_cost_usd",
]
print(
    hub_audit[cols_hub_view].to_string(
        index=False,
        formatters={
            "total_flights": "{:,}".format,
            "primary_triggers": "{:,}".format,
            "induced_flights": "{:,}".format,
            "cascade_multiplier": "{:.3f}x".format,
            "primary_delay_min": "{:,.0f} min".format,
            "induced_delay_min": "{:,.0f} min".format,
            "induced_cost_usd": "${:,.0f}".format,
        },
    )
)

# ==============================================================================
# 2. DENNÍ ČASOVÁ OKNA: KDY KASKÁDA ESKALUJE (DOPAD DLE HODINY V UTC)
# ==============================================================================
# Extrakce plánované hodiny odletu v UTC
df["dep_hour_utc"] = df["CRSDep_dt_utc"].dt.hour

hourly_trend = (
    df.groupby("dep_hour_utc")
    .agg(
        total_flights=("DepDelay", "count"),
        primary_triggers=("is_primary_weather_trigger", "sum"),
        induced_flights=("is_induced_delay", "sum"),
        induced_delay_min=("induced_delay_min", "sum"),
    )
    .reset_index()
)

# Výběr 5 nejkritičtějších hodin dne podle objemu sekundárního zpoždění
top_hours = hourly_trend.sort_values(by="induced_delay_min", ascending=False).head(5)

print("\n=== TOP 5 NEJKRITIČTĚJŠÍCH HODIN DNE PRO VZNIK KASKÁD (UTC) ===")
print(
    top_hours.to_string(
        index=False,
        formatters={
            "total_flights": "{:,}".format,
            "primary_triggers": "{:,}".format,
            "induced_flights": "{:,}".format,
            "induced_delay_min": "{:,.0f} min".format,
        },
    )
)

=== SROVNÁNÍ ODOLNOSTI HUBŮ (KORD vs. KATL vs. KDFW vs. KJFK) ===
Origin total_flights primary_triggers induced_flights cascade_multiplier primary_delay_min induced_delay_min induced_cost_usd
   ORD        21,231            1,508             172             0.094x       142,898 min        13,486 min       $1,348,600
   ATL        28,329            1,586             173             0.085x       110,361 min         9,341 min         $934,100
   DFW        24,780              262             119             0.432x        20,435 min         8,823 min         $882,300
   JFK         9,892              561              65             0.095x        70,397 min         6,667 min         $666,700

=== TOP 5 NEJKRITIČTĚJŠÍCH HODIN DNE PRO VZNIK KASKÁD (UTC) ===
 dep_hour_utc total_flights primary_triggers induced_flights induced_delay_min
            1         5,327              260              77         6,354 min
           23         5,586              385              64         5,023 min
  

### Interpretace 4. části analýzy:
* **Odolnost hubů (DFW vs. ORD/ATL/JFK):**
  * Letiště **KDFW** prokázalo nejnižší schopnost absorbovat špatné počasí – dosáhlo kaskádového indexu **0,432×** (téměř čtyřnásobek průměru sítě). Na každou minutu primárního zpoždění vygeneroval Dallas 0,43 minuty indukovaného skluzu.
  * Největší finanční škodu utrpěl uzel **KORD** ($1,35 mil. USD / 13 486 min indukovaného zpoždění) následovaný **KATL** ($934 tis. USD).
* **Kritické denní okno (Odpolední/večerní vlna):**
  * Největší kumulace sekundárních zpoždění nastává mezi **22:00 a 02:00 UTC** (17:00–21:00 lokálního času), s absolutním vrcholem v **01:00 UTC** (6 354 minut).
  * Toto zpoždění je přímým dozvukem odpolední konvektivní činnosti (14:00–17:00 lokálně), kdy se zpožděné stroje vracejí do uzlů na večerní odletovou vlnu.